# Slices and Multilayer Workflows

`AnnNet` has two related organizational concepts:

- `slices`: named subsets of graph membership, useful for conditions, experiments, or time points
- `layers`: Kivela-style multilayer structure, useful when vertices can exist in several aspect combinations

This notebook shows how they relate and how to move between them.


In [1]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
PACKAGE_ROOT = (NOTEBOOK_DIR / "..").resolve()
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT))

import annnet as an


In [2]:
G = an.AnnNet(directed=True)
G.add_vertices(["TP53", "MDM2", "BAX", "ATM"])

G.slices.add("baseline", condition="baseline")
G.slices.add("treated", condition="dna_damage")

G.add_edge("ATM", "TP53", slice="treated", edge_id="damage_sensing")
G.add_edge("TP53", "MDM2", slice="baseline", edge_id="feedback_baseline")
G.add_edge("TP53", "BAX", slice="treated", edge_id="apoptosis_program")


'apoptosis_program'

In [3]:
print("active slice:", G.slices.active)
print("slices:", G.slices.list(include_default=True))
print("treated info:", G.slices.info("treated"))
print("treated edges:", G.slices.edges("treated"))
G.slices_view()


active slice: default
slices: ['default', 'baseline', 'treated']
treated info: {'vertices': {'ATM', 'TP53', 'BAX'}, 'edges': {'damage_sensing', 'apoptosis_program'}, 'attributes': {'condition': 'dna_damage'}}
treated edges: {'damage_sensing', 'apoptosis_program'}


slice_id,condition
str,str
"""baseline""","""baseline"""
"""treated""","""dna_damage"""


In [4]:
G.slices.union(["baseline", "treated"])


{'vertices': {'ATM', 'BAX', 'MDM2', 'TP53'},
 'edges': {'apoptosis_program', 'damage_sensing', 'feedback_baseline'}}

## Configure a single-aspect multilayer structure

Here the aspect is `condition`, with two elementary layers: `baseline` and `treated`.


In [5]:
G.set_aspects(["condition"], {"condition": ["baseline", "treated"]})

for layer in [("baseline",), ("treated",)]:
    for vertex in ["TP53", "MDM2", "BAX", "ATM"]:
        G.add_presence(vertex, layer)

G.add_intra_edge("TP53", "MDM2", "baseline", eid="ml_feedback")
G.add_intra_edge("TP53", "BAX", "treated", eid="ml_apoptosis")
G.add_inter_edge("TP53", "TP53", "baseline", "treated", eid="tp53_state_transfer")


'tp53_state_transfer'

In [6]:
print("aspects:", G.layers.aspects())
print("elementary layers:", G.layers.elementary_layers())
print("layer tuples:", G.layers.layer_tuples())
print("baseline vertices:", G.layers.vertex_set(("baseline",)))
print("baseline edges:", G.layers.edge_set(("baseline",), include_inter=True))


aspects: ['condition']
elementary layers: {'condition': ['baseline', 'treated']}
layer tuples: [('baseline',), ('treated',)]
baseline vertices: {'TP53', 'ATM', 'MDM2', 'BAX'}
baseline edges: {'tp53_state_transfer', 'ml_feedback'}


In [7]:
G.layers.to_slice(("treated",), slice_id="treated_from_layers", source="layers API")
print(G.slices.info("treated_from_layers"))


{'vertices': {'TP53', 'ATM', 'MDM2', 'BAX'}, 'edges': {'ml_apoptosis'}, 'attributes': {'source': 'layers API', 'layer_tuple': ('treated',)}}


In [8]:
A = G.layers.supra_adjacency()
print(type(A).__name__)
print(A.shape)


csr_matrix
(8, 8)
